# Large-scale EV module/busbar detector (YOLO11n, GPU)

Trains a **generalist** on the diverse multi-source dataset (10 real EV pack
sources, 4,478 train imgs) and evaluates on a **diverse 225-image test set**
spanning all sources (leakage-cleaned). This is the honest benchmark for
*detecting EV modules at scale*, not the narrow single-facility MTech test.

**Number to beat:** the old MTech specialist scores only **0.277 mAP50** on this
diverse test (it only knows one pack style). A generalist should crush that.

YOLO11n is smaller (2.6M vs 3.2M params) *and* typically more accurate than v8n.

Upload `ev_diverse_data.zip` (in ~/Downloads) to Drive anywhere; the notebook finds it.

In [ ]:
import torch, subprocess
print('CUDA:', torch.cuda.is_available(), '|', subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader'))
!pip install -q ultralytics

In [ ]:
from google.colab import drive
import glob, subprocess
drive.mount('/content/drive')
hits = glob.glob('/content/drive/MyDrive/**/ev_diverse_data.zip', recursive=True)
assert hits, 'ev_diverse_data.zip not found in Drive — upload it first.'
print('Found:', hits[0])
subprocess.run(['unzip','-q','-o',hits[0],'-d','/content/'])
!echo 'train:' $(ls /content/ev_diverse/images/train/*.jpg | wc -l) ' test:' $(ls /content/ev_diverse/images/test/*.jpg | wc -l)

In [ ]:
# Train YOLO11n on the diverse multi-source data
from ultralytics import YOLO
YAML = '/content/ev_diverse/dataset.yaml'
model = YOLO('yolo11n.pt')
model.train(data=YAML, epochs=150, imgsz=640, batch=32, device=0,
            optimizer='SGD', lr0=0.01, weight_decay=0.0005,
            hsv_v=0.30, hsv_s=0.25, hsv_h=0.0, mosaic=0.5,
            degrees=2.0, translate=0.06, fliplr=0.5, patience=40,
            project='/content/runs', name='largescale', exist_ok=True)

In [ ]:
# Evaluate on the diverse 225-image test set (plain + TTA)
print('=== diverse test, plain ===')
m = model.val(data=YAML, split='test', imgsz=640, device=0, name='ls_test', project='/content/runs', exist_ok=True)
print(f'  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}   (MTech specialist got 0.277 here)')
for i,c in m.names.items(): print(f'    {c}: mAP50={m.box.ap50[i]:.3f}')
print('=== diverse test + TTA ===')
mt = model.val(data=YAML, split='test', imgsz=640, device=0, augment=True, name='ls_test_tta', project='/content/runs', exist_ok=True)
print(f'  mAP50={mt.box.map50:.3f}  mAP50-95={mt.box.map:.3f}')

In [ ]:
from google.colab import files
best='/content/runs/largescale/weights/best.pt'
!cp "$best" /content/drive/MyDrive/ev_largescale_yolo11n_best.pt
print('Saved to Drive: MyDrive/ev_largescale_yolo11n_best.pt'); files.download(best)

## Read the result
This is the real goal: a model that detects EV modules **across many pack types**.
- Compare the diverse-test mAP50 to the specialist's **0.277**. A big jump = the
  generalist works at scale.
- Send me the numbers + download `best.pt`; I'll run per-source breakdowns and we
  decide next steps (auto-labeling to grow it further, ensembling, etc.).